# Connect Your LLM to Google Sheets — Zero to Hero

This notebook teaches production-grade hybrid analytics:
- Google Sheets ingestion
- deterministic Python analytics
- local LLM business insights
- conversational analytics and reporting

## Why deterministic decoding (`temperature=0`)

Business analytics must be reproducible and auditable.
With `temperature=0`, the same prompt and evidence produce stable outputs,
which improves consistency for benchmarking, QA, and stakeholder trust.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from ai_spreadsheet_analytics.config import Settings
settings = Settings()
settings.ensure_directories()
settings

## Authentication: Service Account vs OAuth

- Service Account: recommended for production pipelines and shared team sheets.
- OAuth: useful for user-scoped access in interactive personal workflows.

Runtime default in this project is Service Account.

In [ ]:
from ai_spreadsheet_analytics.connectors.auth import build_service_account_client
from ai_spreadsheet_analytics.connectors.google_sheets import GoogleSheetsLoader, SheetLoadRequest
from ai_spreadsheet_analytics.state_store import SQLiteStateStore

state_store = SQLiteStateStore(settings.state_db_path)
if settings.google_service_account_json and settings.google_service_account_json.exists():
    client = build_service_account_client(settings.google_service_account_json, settings.scopes)
    loader = GoogleSheetsLoader(client=client, cache_dir=settings.cache_dir, state_store=state_store)
    print('Google Sheets client ready')
else:
    loader = None
    print('Google credentials missing. Live Sheets steps skipped; continue with offline CSV path.')


In [ ]:
# Example live load (uncomment and set IDs)
# requests = [
#     SheetLoadRequest(spreadsheet_id='YOUR_SHEET_ID', worksheet_title='sales_transactions'),
#     SheetLoadRequest(spreadsheet_id='YOUR_SHEET_ID', worksheet_title='product_master'),
# ]
# bundle = loader.load_batch(requests)
# df = bundle.combined()
# df.head()

## Offline learning path with sample CSV
Use this when live credentials are not configured yet.

In [ ]:
import pandas as pd
from pathlib import Path
from ai_spreadsheet_analytics.quality import DataQualityProfiler
from ai_spreadsheet_analytics.cleaning import DataCleaner
from ai_spreadsheet_analytics.schemas import CleaningStrategy
from ai_spreadsheet_analytics.analytics import AnalyticsEngine

cwd = Path.cwd()
candidates = [
    cwd / 'data/samples/retail_sales.csv',
    cwd.parent / 'data/samples/retail_sales.csv',
]
csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f'Retail sample CSV not found. Checked: {candidates}')

df = pd.read_csv(csv_path)
quality = DataQualityProfiler().profile('retail_sales', df)
cleaned = DataCleaner().clean('retail_sales', df, CleaningStrategy(missing_value_strategy='median'))
eda = AnalyticsEngine().run_full_eda(cleaned.cleaned)
quality.metrics, cleaned.actions[:3], list(eda.keys())


## LLM Insights (Direct REST)

In [ ]:
from ai_spreadsheet_analytics.llm.ollama_rest import OllamaRESTClient
from ai_spreadsheet_analytics.insights import InsightEngine

llm = InsightEngine(OllamaRESTClient(settings.ollama_base_url), default_temperature=0.0)
# packet = llm.generate(analytics_payload=eda, role='executive', model=settings.ollama_primary_model)
# print(packet.summary)

## Compare Direct Ollama API vs LangChain orchestration

In [ ]:
from ai_spreadsheet_analytics.llm_compare import compare_ollama_paths_sync

# comparison = compare_ollama_paths_sync(
#     prompt='Summarize monthly revenue trend from evidence.',
#     model=settings.ollama_primary_model,
#     runs=3,
#     base_url=settings.ollama_base_url,
# )
# comparison

## Conversational Analytics

In [ ]:
from ai_spreadsheet_analytics.chat import ConversationalAnalyticsAssistant

assistant = ConversationalAnalyticsAssistant(
    analytics_engine=AnalyticsEngine(),
    insight_engine=llm,
    state_store=state_store,
    model=settings.ollama_primary_model,
)

# turn = assistant.ask('Which products sold best?', cleaned.cleaned, session_id='demo-notebook')
# turn.answer

## Report Generation

In [ ]:
from ai_spreadsheet_analytics.reporting import ReportGenerator

reporter = ReportGenerator(settings.report_dir)
# artifacts = reporter.generate(
#     title='Retail Executive Report',
#     insights_markdown=packet.summary,
#     tables={
#         'cleaned_preview': cleaned.cleaned.head(50),
#         'numeric_summary': pd.DataFrame(eda['summary']['numeric_summary']),
#     },
# )
# artifacts

## Next Steps
1. Connect live Google Sheets and run ingestion.
2. Execute benchmark suite.
3. Run Streamlit dashboard.
4. Run judge evaluations and compare models.